In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Seed for reproducibility
np.random.seed(42)

# Total records needed
num_records = 520 

# Base Data Options
routes = {
    "R-101": "Route A (Downtown - Airport)",
    "R-102": "Route B (Saddar - Gulberg)",
    "R-103": "Route C (Johar Town - Mall Road)",
    "R-104": "Route D (Clifton - DHA)",
    "R-105": "Route E (Metro Hub - Industrial Area)"
}
bus_types = ["Double Decker", "Standard Electric", "Articulated Bus", "Mini Bus"]
weathers = ["Clear", "Rainy", "Foggy", "Overcast"]
trip_statuses = ["On Time", "Delayed", "Delayed", "Cancelled", "On Time"] # weighted for realism

route_ids = np.random.choice(list(routes.keys()), size=num_records)
route_names = [routes[rid] for rid in route_ids]

# Generating columns
data = {
    "Route_ID": route_ids,
    "Route_Name": route_names,
    "Bus_Number": [f"BUS-{np.random.randint(1000, 9999)}" for _ in range(num_records)],
    "Bus_Type": np.random.choice(bus_types, size=num_records),
    "Date": [(datetime(2026, 6, 1) + timedelta(days=int(np.random.randint(0, 30)))).strftime('%Y-%m-%d') for _ in range(num_records)],
    "Departure_Time": [f"{np.random.randint(6, 23):02d}:{np.random.choice([0, 15, 30, 45]):02d}" for _ in range(num_records)],
    "Passenger_Count": np.random.randint(10, 85, size=num_records),
    "Ticket_Revenue": lambda x: x["Passenger_Count"] * np.random.choice([120, 150, 200]), # placeholder handle below
    "Fuel_Consumption_Liters": np.round(np.random.uniform(15.5, 45.0, size=num_records), 2),
    "Driver_ID": [f"DRV-{np.random.randint(100, 999)}" for _ in range(num_records)],
    "Weather_Condition": np.random.choice(weathers, size=num_records),
    "Trip_Status": np.random.choice(trip_statuses, size=num_records)
}

df = pd.DataFrame(data)

# Dynamic logical connections
df["Ticket_Revenue"] = df["Passenger_Count"] * np.random.choice([80, 100, 120], size=num_records)
df["Delay_Duration_Minutes"] = np.where(df["Trip_Status"] == "Delayed", np.random.randint(10, 65, size=num_records), 0)
df["Delay_Duration_Minutes"] = np.where(df["Trip_Status"] == "Cancelled", 0, df["Delay_Duration_Minutes"])

# Day Type Calculation
df["Day_Type"] = pd.to_datetime(df["Date"]).dt.dayofweek.map(lambda x: "Weekend" if x >= 5 else "Weekday")

# Reordering columns to match requirements perfectly
columns_order = [
    "Route_ID", "Route_Name", "Bus_Number", "Bus_Type", "Date", 
    "Departure_Time", "Delay_Duration_Minutes", "Passenger_Count", 
    "Ticket_Revenue", "Fuel_Consumption_Liters", "Driver_ID", 
    "Weather_Condition", "Day_Type", "Trip_Status"
]
df = df[columns_order]

# Introduce minimal missing values to show "Data Cleaning" ability
for col in ["Delay_Duration_Minutes", "Fuel_Consumption_Liters"]:
    df.loc[df.sample(frac=0.02).index, col] = np.nan

# Save to CSV
df.to_csv("Smart_Public_Transport_Dataset.csv", index=False)
print("Dataset created successfully with 520 rows and 14 columns!")

Dataset created successfully with 520 rows and 14 columns!


In [2]:
# STEP 2: DATA CLEANING & VALIDATION
# =====================================================================

print("--- Checking for Missing Values Before Cleaning ---")
print(df.isnull().sum())

# 1. Handling Missing Values
# Fill missing delay minutes with 0 (assuming no delay if missing)
df["Delay_Duration_Minutes"] = df["Delay_Duration_Minutes"].fillna(0)

# Fill missing fuel consumption with the median value of that specific Bus Type
df["Fuel_Consumption_Liters"] = df.groupby("Bus_Type")["Fuel_Consumption_Liters"].transform(lambda x: x.fillna(x.median()))

# 2. Data Type Correction & Standardization
df["Date"] = pd.to_datetime(df["Date"])
df["Delay_Duration_Minutes"] = df["Delay_Duration_Minutes"].astype(int)
df["Passenger_Count"] = df["Passenger_Count"].astype(int)
df["Ticket_Revenue"] = df["Ticket_Revenue"].astype(float)

# 3. Duplicate Removal
initial_shape = df.shape
df = df.drop_duplicates()
print(f"\nRemoved {initial_shape[0] - df.shape[0]} duplicate rows.")

# 4. Outlier Detection & Outlier Handling (Cap extreme values if any)
# Capping high delays to a realistic maximum of 120 minutes
df["Delay_Duration_Minutes"] = np.where(df["Delay_Duration_Minutes"] > 120, 120, df["Delay_Duration_Minutes"])

print("\n--- Checking for Missing Values After Cleaning ---")
print(df.isnull().sum())

# Save Cleaned Data to a separate CSV for the Dashboard
df.to_csv("Cleaned_Smart_Public_Transport_Data.csv", index=False)
print("\nCleaned dataset saved successfully as 'Cleaned_Smart_Public_Transport_Data.csv'!")

--- Checking for Missing Values Before Cleaning ---
Route_ID                    0
Route_Name                  0
Bus_Number                  0
Bus_Type                    0
Date                        0
Departure_Time              0
Delay_Duration_Minutes     10
Passenger_Count             0
Ticket_Revenue              0
Fuel_Consumption_Liters    10
Driver_ID                   0
Weather_Condition           0
Day_Type                    0
Trip_Status                 0
dtype: int64

Removed 0 duplicate rows.

--- Checking for Missing Values After Cleaning ---
Route_ID                   0
Route_Name                 0
Bus_Number                 0
Bus_Type                   0
Date                       0
Departure_Time             0
Delay_Duration_Minutes     0
Passenger_Count            0
Ticket_Revenue             0
Fuel_Consumption_Liters    0
Driver_ID                  0
Weather_Condition          0
Day_Type                   0
Trip_Status                0
dtype: int64

Cleaned dataset 

In [3]:
# =====================================================================
# STEP 3: EXPLORATORY DATA ANALYSIS (EDA) & KPI CALCULATIONS
# =====================================================================

print("--- 1. OVERALL BUSINESS KEY PERFORMANCE INDICATORS (KPIs) ---")
total_passengers = df["Passenger_Count"].sum()
total_revenue = df["Ticket_Revenue"].sum()
avg_delay = df["Delay_Duration_Minutes"].mean()
total_trips = len(df)
cancelled_trips = len(df[df["Trip_Status"] == "Cancelled"])
completion_rate = ((total_trips - cancelled_trips) / total_trips) * 100

print(f"Total Passengers Transported: {total_passengers:,}")
print(f"Total Ticket Revenue Generated: Rs {total_revenue:,.2f}")
print(f"Average Delay Time: {avg_delay:.2f} Minutes")
print(f"Daily Trip Completion Rate: {completion_rate:.2f}%\n")

print("--- 2. ROUTE PERFORMANCE ANALYTICS (Top Revenue & Passengers) ---")
route_perf = df.groupby("Route_Name").agg(
    Total_Passengers=("Passenger_Count", "sum"),
    Total_Revenue=("Ticket_Revenue", "sum"),
    Average_Delay=("Delay_Duration_Minutes", "mean")
).sort_values(by="Total_Revenue", ascending=False)
print(route_perf)

print("\n--- 3. TRIP STATUS DISTRIBUTION ---")
print(df["Trip_Status"].value_counts())

print("\n--- 4. WEATHER IMPACT ON DELAYS ---")
weather_delay = df.groupby("Weather_Condition")["Delay_Duration_Minutes"].mean().sort_values(ascending=False)
print(weather_delay)

--- 1. OVERALL BUSINESS KEY PERFORMANCE INDICATORS (KPIs) ---
Total Passengers Transported: 24,920
Total Ticket Revenue Generated: Rs 2,501,000.00
Average Delay Time: 15.47 Minutes
Daily Trip Completion Rate: 78.65%

--- 2. ROUTE PERFORMANCE ANALYTICS (Top Revenue & Passengers) ---
                                       Total_Passengers  Total_Revenue  \
Route_Name                                                               
Route A (Downtown - Airport)                       5754       572760.0   
Route D (Clifton - DHA)                            5483       547760.0   
Route E (Metro Hub - Industrial Area)              4642       462620.0   
Route B (Saddar - Gulberg)                         4585       459640.0   
Route C (Johar Town - Mall Road)                   4456       458220.0   

                                       Average_Delay  
Route_Name                                            
Route A (Downtown - Airport)               16.601770  
Route D (Clifton - DHA)          

In [4]:
from IPython.display import FileLink
FileLink('Cleaned_Smart_Public_Transport_Data.csv')

C:\Users\My Pc\Downloads\Cleaned_Smart_Public_Transport_Data.csv